# 05b · Modèle 2 : corrections et modèle final (risque de retard)
**Projet : E-Commerce Intelligence Platform** · CRISP-DM, phase 5 (Evaluation), avant le déploiement

**Pourquoi ce notebook ?** Le rapport d'évaluation a montré que le classifieur **perd presque toute sa performance entre valid et test** (ROC-AUC de 0,75 à 0,64), que la **régression puis seuil** résiste mieux (ROC-AUC de 0,77 sur test), et que les alertes **ne sont pas rentables** avec les coûts testés. Il signale aussi que la comparaison « top 10 % » n'était pas à capacité égale.

**Ce que fait ce notebook**
1. **Six approches** comparées : classifieur avec toutes les variables, classifieur **sans proxys du temps** (`orders_last_7d`, `purchase_month`), **régression L1** (score = durée prédite − promesse), **régression quantile 0,8**, **classifieur + écart de durée** (écart calculé hors échantillon, sans fuite), et règle « distance / promesse ».
2. **Évaluation glissante par mois** : chaque mois est prédit par des modèles entraînés sur les mois précédents (ROC-AUC, PR-AUC et gain par rapport au taux de base).
3. **Évaluation à capacité égale** : on signale exactement les 5 %, 10 % ou 20 % de commandes les plus risquées *du test* et on mesure précision, rappel et gain.
4. **Rentabilité** : bénéfice net selon le coût d'un retard manqué, la part de retards réellement évités et la capacité.
5. **Modèle final** et fichiers de déploiement.

> Données : datasets Kaggle `e-commerce-dataset-model-2` et `e-commerce-dataset-model-1` (pour `delivery_days`, utilisé **uniquement** comme cible du régresseur). Durée indicative : 10 à 25 minutes. Fichiers écrits dans `/kaggle/working/deploy/`.

## 1. Imports et paramètres

In [ ]:
import os, glob, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

SEED = 42
FAST = os.environ.get("FAST", "0") == "1"
OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working/deploy")
os.makedirs(OUT_DIR, exist_ok=True)

PROXY = ["orders_last_7d", "purchase_month"]
N_EST = 100 if FAST else 300
OOF_SPLITS = 3 if FAST else 4
QUANT_ALPHA = 0.8
ROLL_MIN_TRAIN = 200 if FAST else 5000
TOPK = [0.05, 0.10, 0.20]                  # capacités d'action testées
COST_FN_GRID = [3, 5, 10, 20, 30]          # coût d'un retard non traité (unité arbitraire)
COST_FP = 1.0                              # coût d'une fausse alerte
PREVENT_RATES = [1.0, 0.5]                 # part des retards alertés qui sont réellement évités
FINAL_APPROACH = None                      # None : choix automatique ; sinon un nom du tableau

# Chemins des jeux de données Kaggle (créés à partir de la sortie du notebook 03)
DATA_PATHS = {
    "model1_delivery_duration.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-1/model1_delivery_duration.csv",
    "model2_late_risk.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-2/model2_late_risk.csv",
    "model3_customer_segmentation.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-3/model3_customer_segmentation.csv",
}

def find_file(name):
    """DATA_DIR (si défini), puis le chemin Kaggle ci-dessus, puis une recherche dans /kaggle/input."""
    cands = []
    if os.environ.get("DATA_DIR"):
        cands.append(os.path.join(os.environ["DATA_DIR"], name))
    cands.append(DATA_PATHS.get(name, ""))
    cands += glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    for c in cands:
        if c and os.path.exists(c):
            return c
    raise FileNotFoundError(f"{name} introuvable : ajoute le dataset correspondant en entrée du notebook (Add Input).")

## 2. Chargement

In [ ]:
m2 = pd.read_csv(find_file("model2_late_risk.csv"), parse_dates=["order_purchase_timestamp"])
m1y = pd.read_csv(find_file("model1_delivery_duration.csv"), usecols=["order_id", "delivery_days"])
m2 = m2.merge(m1y, on="order_id", how="left")       # delivery_days : cible du régresseur d'écart, jamais une variable explicative

FEATURES_NUMERIC = [
    "nb_items", "multi_seller", "total_price", "total_freight", "total_weight_g", "total_volume_cm3",
    "weight_imputed", "freight_ratio", "limit_days", "distance_km", "distance_imputed", "same_state",
    "customer_lat", "customer_lng", "seller_lat", "seller_lng", "payment_installments",
    "purchase_month", "purchase_dow", "purchase_hour", "is_weekend", "is_black_friday_week", "orders_last_7d",
    "seller_n_past", "seller_mean_days", "seller_late_rate", "route_n_past", "route_mean_days", "route_late_rate",
    "log_total_price", "log_total_freight", "log_total_weight_g", "log_total_volume_cm3", "log_distance_km",
]
CATS = ["customer_state", "seller_state", "categorie_grouped", "payment_type"]
PROMISE = ["promised_days"]
FEATURES_CORE = FEATURES_NUMERIC + CATS

needed = FEATURES_CORE + PROMISE + ["is_late", "split", "order_id", "delivery_days", "customer_state"]
assert not [c for c in needed if c not in m2.columns], "Colonnes absentes"
assert m2["delivery_days"].notna().all(), "delivery_days manquant : vérifier que le dataset du modèle 1 correspond"

full = m2.sort_values("order_purchase_timestamp").reset_index(drop=True)
full["month"] = full["order_purchase_timestamp"].dt.to_period("M").astype(str)
roll_months = sorted(full.loc[full["split"].isin(["valid", "test"]), "month"].unique())
print(f"{len(full):,} commandes | mois évalués en glissant : {roll_months}")
print(full.groupby("split")["is_late"].agg(["count", "mean"]).reindex(["train", "valid", "test"]))

## 3. Les six approches

In [ ]:
def make_converter(feats, ref):
    cats = [c for c in CATS if c in feats]
    levels = {c: sorted(ref[c].astype(str).unique()) for c in cats}
    def conv(df):
        X = df[feats].copy()
        for c in cats:
            X[c] = pd.Categorical(X[c].astype(str), categories=levels[c])
        return X
    conv.levels = levels
    return conv

LGB = dict(n_estimators=N_EST, learning_rate=0.05, num_leaves=31, min_child_samples=50, subsample=0.8, subsample_freq=1,
           colsample_bytree=0.7, reg_lambda=5.0, random_state=SEED, n_jobs=-1, verbose=-1)
FEATS_FULL = FEATURES_CORE + PROMISE
FEATS_NP = [f for f in FEATS_FULL if f not in PROXY]

def clf_fit(trn, feats, y=None):
    conv = make_converter(feats, trn)
    yy = trn["is_late"].values if y is None else y
    ratio = (yy == 0).sum() / max((yy == 1).sum(), 1)
    return lgb.LGBMClassifier(**{**LGB, "scale_pos_weight": float(np.sqrt(ratio))}).fit(conv(trn), yy), conv

def reg_fit(trn, feats, objective="regression_l1", alpha=None):
    conv = make_converter(feats, trn)
    p = {**LGB, "objective": objective}
    if alpha is not None:
        p["alpha"] = alpha
    return lgb.LGBMRegressor(**p).fit(conv(trn), trn["delivery_days"].values), conv

def oof_gap(trn, feats):
    """Écart durée prédite - promesse, calculé hors échantillon (blocs temporels successifs) : aucune fuite.
    Le premier bloc n'a pas d'historique : son écart reste manquant (LightGBM gère les NaN)."""
    gap = np.full(len(trn), np.nan)
    for a, b in TimeSeriesSplit(n_splits=OOF_SPLITS).split(np.arange(len(trn))):
        m, conv = reg_fit(trn.iloc[a], feats)
        gap[b] = m.predict(conv(trn.iloc[b])) - trn["promised_days"].values[b]
    return gap

A1, A2, A3, A4, A5, A6 = ("Classifieur (toutes variables)", "Classifieur (sans proxys du temps)", "Régression L1 : écart durée - promesse",
                          "Régression quantile 0,8 : écart", "Classifieur + écart de durée", "Règle : distance / promesse")
APPROACHES = [A1, A2, A3, A4, A5, A6]

def fit_approach(name, trn):
    """Entraîne une approche ; renvoie (fonction de score, artefacts). Plus le score est élevé, plus le risque de retard est fort."""
    if name == A1:
        m, c = clf_fit(trn, FEATS_FULL); return (lambda d: m.predict_proba(c(d))[:, 1]), {"clf": m, "levels": c.levels, "feats": FEATS_FULL}
    if name == A2:
        m, c = clf_fit(trn, FEATS_NP); return (lambda d: m.predict_proba(c(d))[:, 1]), {"clf": m, "levels": c.levels, "feats": FEATS_NP}
    if name == A3:
        r, c = reg_fit(trn, FEATS_NP); return (lambda d: r.predict(c(d)) - d["promised_days"].values), {"reg": r, "levels": c.levels, "feats": FEATS_NP}
    if name == A4:
        r, c = reg_fit(trn, FEATS_NP, "quantile", QUANT_ALPHA)
        return (lambda d: r.predict(c(d)) - d["promised_days"].values), {"reg": r, "levels": c.levels, "feats": FEATS_NP}
    if name == A5:
        g = oof_gap(trn, FEATS_NP); r, rc = reg_fit(trn, FEATS_NP)
        trn2 = trn.assign(gap=g); feats2 = FEATS_NP + ["gap"]
        m, c2 = clf_fit(trn2, feats2)
        def fn(d):
            d2 = d.assign(gap=r.predict(rc(d)) - d["promised_days"].values)
            return m.predict_proba(c2(d2))[:, 1]
        return fn, {"clf": m, "reg": r, "levels": c2.levels, "feats": feats2}
    if name == A6:
        return (lambda d: (d["distance_km"] / d["promised_days"]).values), {}
    raise ValueError(name)

def pr_auc(y, s): return float(average_precision_score(y, s))
def roc_auc(y, s): return float(roc_auc_score(y, s))
print("Approches :", APPROACHES)

## 4. Évaluation glissante par mois
Pour chaque mois, toutes les approches sont entraînées sur **les commandes antérieures au 1er du mois** et évaluées sur le mois. Indicateurs : ROC-AUC, PR-AUC et **gain** (PR-AUC divisée par le taux de retard du mois, ce qui rend les mois comparables malgré les taux de base très différents).

In [ ]:
roll_rows = []
t0 = time.time()
for mth in roll_months:
    start = pd.Period(mth, "M").start_time
    trn = full[full["order_purchase_timestamp"] < start]; tst = full[full["month"] == mth]
    if len(trn) < ROLL_MIN_TRAIN or len(tst) == 0 or tst["is_late"].nunique() < 2:
        print("  mois ignoré :", mth); continue
    y = tst["is_late"].values
    for name in APPROACHES:
        fn, _ = fit_approach(name, trn)
        s = fn(tst)
        roll_rows.append({"mois": mth, "approche": name, "n": len(y), "taux_retard": y.mean(),
                          "ROC-AUC": roc_auc(y, s), "PR-AUC": pr_auc(y, s), "gain": pr_auc(y, s) / y.mean()})
    print(f"  {mth} : {len(trn):,} commandes d'entraînement, {len(tst):,} évaluées ({time.time() - t0:.0f} s)")
roll = pd.DataFrame(roll_rows)

## 5. Résultats de l'évaluation glissante

In [ ]:
def wmean(g, col): return np.average(g[col], weights=g["n"])
agg = (roll.groupby("approche").apply(lambda g: pd.Series({"ROC-AUC moyen": wmean(g, "ROC-AUC"), "PR-AUC moyenne": wmean(g, "PR-AUC"),
                                                            "gain moyen": wmean(g, "gain"), "ROC-AUC min": g["ROC-AUC"].min()}))
         .sort_values("gain moyen", ascending=False))
display(agg)
for col in ["ROC-AUC", "gain"]:
    print(f"\n{col} par mois :")
    display(roll.pivot(index="mois", columns="approche", values=col))

fig, ax = plt.subplots(1, 2, figsize=(18, 5))
for name in APPROACHES:
    d = roll[roll["approche"] == name]
    ax[0].plot(d["mois"], d["ROC-AUC"], marker="o", label=name); ax[1].plot(d["mois"], d["gain"], marker="o", label=name)
ax[0].axhline(0.5, color="k", ls=":"); ax[0].set_title("ROC-AUC par mois (0,5 = hasard)"); ax[1].axhline(1, color="k", ls=":")
ax[1].set_title("Gain de PR-AUC par rapport au taux de base"); ax[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

a1, a2 = agg.loc[A1, "ROC-AUC moyen"], agg.loc[A2, "ROC-AUC moyen"]
print(f"Effet du retrait de orders_last_7d et purchase_month sur le classifieur : ROC-AUC moyen {a1:.3f} -> {a2:.3f} ({a2 - a1:+.3f})")
print(f"Régression + écart de durée contre classifieur direct (ROC-AUC moyen) : {agg.loc[A3, 'ROC-AUC moyen']:.3f} contre {a2:.3f}")

## 6. Choix de l'approche finale
On retient l'approche de **gain moyen maximal** parmi les approches entraînées (la règle « distance / promesse » reste un point de comparaison). `FINAL_APPROACH` permet d'en forcer une autre.

In [ ]:
cands = agg.drop(index=[A6], errors="ignore")
final_approach = FINAL_APPROACH if FINAL_APPROACH else cands.index[0]
print("Approche retenue :", final_approach)
display(agg.loc[[final_approach, A6]])

## 7. Évaluation statique à capacité égale (train + valid → test)
Toutes les approches sont ajustées sur train + valid, puis évaluées **une fois** sur le test. On signale exactement les 5 %, 10 % et 20 % de commandes les **plus risquées du test**, ce qui rend la comparaison équitable entre approches.

In [ ]:
trva = full[full["split"].isin(["train", "valid"])]; tst = full[full["split"] == "test"]
yte = tst["is_late"].values; base_rate = yte.mean()
scores_te, arts = {}, {}
for name in APPROACHES:
    fn, art = fit_approach(name, trva)
    scores_te[name], arts[name] = fn(tst), art

def topk_stats(y, s, k):
    n = max(int(round(len(y) * k)), 1)
    idx = np.argsort(-s)[:n]
    tp = y[idx].sum()
    return {"précision": tp / n, "rappel": tp / max(y.sum(), 1), "gain": (tp / n) / max(y.mean(), 1e-9), "TP": int(tp), "FP": int(n - tp)}

rows = []
for name in APPROACHES:
    s = scores_te[name]
    r = {"approche": name, "ROC-AUC": roc_auc(yte, s), "PR-AUC": pr_auc(yte, s), "gain PR-AUC": pr_auc(yte, s) / base_rate}
    for k in TOPK:
        t = topk_stats(yte, s, k)
        r[f"précision top {int(k * 100)} %"] = t["précision"]; r[f"rappel top {int(k * 100)} %"] = t["rappel"]
    rows.append(r)
static = pd.DataFrame(rows).set_index("approche").sort_values("gain PR-AUC", ascending=False)
print(f"Taux de retard du test : {base_rate * 100:.2f} %")
display(static)

## 8. Rentabilité des alertes
Bénéfice net par rapport à ne rien faire : `part évitée × coût d'un retard manqué × retards alertés − coût d'une fausse alerte × fausses alertes`. Pour qu'une alerte soit rentable, il faut une précision supérieure à `coût_FP / (coût_FP + part évitée × coût_FN)`. Calcul sur le test, pour l'approche retenue, à capacité égale.

In [ ]:
s_final = scores_te[final_approach]
rows = []
for k in TOPK:
    t = topk_stats(yte, s_final, k)
    for prev in PREVENT_RATES:
        for cfn in COST_FN_GRID:
            net = prev * cfn * t["TP"] - COST_FP * t["FP"]
            rows.append({"capacité": f"top {int(k * 100)} %", "part évitée": prev, "coût retard manqué": cfn,
                         "précision": t["précision"], "précision de rentabilité": COST_FP / (COST_FP + prev * cfn),
                         "bénéfice net": net, "rentable": net > 0})
rent = pd.DataFrame(rows)
display(rent[rent["part évitée"] == 1.0].pivot(index="capacité", columns="coût retard manqué", values="bénéfice net"))
display(rent[rent["part évitée"] == 0.5].pivot(index="capacité", columns="coût retard manqué", values="bénéfice net"))

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
for j, prev in enumerate(PREVENT_RATES):
    d = rent[rent["part évitée"] == prev]
    for cap, g in d.groupby("capacité"):
        ax[j].plot(g["coût retard manqué"], g["bénéfice net"], marker="o", label=cap)
    ax[j].axhline(0, color="k", ls=":"); ax[j].set_xlabel("Coût d'un retard manqué (coût d'une fausse alerte = 1)")
    ax[j].set_ylabel("Bénéfice net"); ax[j].set_title(f"Bénéfice net si {int(prev * 100)} % des retards alertés sont évités"); ax[j].legend()
plt.tight_layout(); plt.show()

be = rent.groupby(["capacité", "part évitée"]).apply(lambda g: g.loc[g["rentable"], "coût retard manqué"].min() if g["rentable"].any() else np.nan)
print("Coût minimal d'un retard manqué à partir duquel l'alerte devient rentable (NaN : jamais dans la grille) :")
display(be.unstack("part évitée"))

## 9. Seuils opérationnels à capacité fixée
Plutôt qu'un seuil de probabilité, qui dérive avec le taux de retard, on définit le seuil par **rang** : on signale les `k` % de commandes les plus risquées de la période. Les seuils ci-dessous (sur le score de l'approche retenue, calculé sur le test) servent de repère ; le dashboard recalculera le rang à chaque période.

In [ ]:
thr = {f"top_{int(k * 100)}": float(np.quantile(s_final, 1 - k)) for k in TOPK}
print("Seuils de score (rang) :", thr)

by_month = tst.assign(score=s_final, month=tst["month"])
rows = []
for mth, g in by_month.groupby("month"):
    y = g["is_late"].values
    if len(y) < 50 or y.sum() == 0:
        continue
    t = topk_stats(y, g["score"].values, 0.10)
    rows.append({"mois": mth, "commandes": len(y), "taux_retard": y.mean(), "précision top 10 %": t["précision"],
                 "rappel top 10 %": t["rappel"], "gain": t["gain"]})
display(pd.DataFrame(rows).set_index("mois"))

## 10. Modèle final et fichiers de déploiement
L'approche retenue est ajustée sur **toutes** les commandes disponibles. Fichiers écrits : composants du modèle, niveaux de catégories, scores et rangs sur le test, tableaux de résultats, manifeste.

In [ ]:
fn_final, art_final = fit_approach(final_approach, full)
joblib.dump({"approach": final_approach, **{k: v for k, v in art_final.items() if k != "levels"}}, os.path.join(OUT_DIR, "model2_components.joblib"))
json.dump({"approach": final_approach, "features": art_final.get("feats", []), "levels": art_final.get("levels", {}),
           "thresholds_rank_on_test": thr, "alpha_quantile": QUANT_ALPHA if final_approach == A4 else None},
          open(os.path.join(OUT_DIR, "model2_config.json"), "w"), ensure_ascii=False, indent=1)

pred = tst[["order_id", "month", "is_late"]].assign(score=s_final)
pred["rang_percentile"] = pred["score"].rank(pct=True)
pred.to_csv(os.path.join(OUT_DIR, "model2_test_scores.csv"), index=False)
roll.to_csv(os.path.join(OUT_DIR, "model2_rolling_results.csv"), index=False)
agg.to_csv(os.path.join(OUT_DIR, "model2_rolling_summary.csv"))
static.to_csv(os.path.join(OUT_DIR, "model2_static_test_topk.csv"))
rent.to_csv(os.path.join(OUT_DIR, "model2_profitability.csv"), index=False)

manifest = {"final_approach": final_approach, "trained_on_rows": int(len(full)), "training_end": str(full["order_purchase_timestamp"].max()),
            "rolling_roc_auc": float(agg.loc[final_approach, "ROC-AUC moyen"]), "rolling_gain": float(agg.loc[final_approach, "gain moyen"]),
            "test_roc_auc_static": float(static.loc[final_approach, "ROC-AUC"]), "cost_fp": COST_FP, "cost_fn_grid": COST_FN_GRID}
json.dump(manifest, open(os.path.join(OUT_DIR, "model2_manifest.json"), "w"), ensure_ascii=False, indent=2)
print("Fichiers de déploiement :", sorted(os.listdir(OUT_DIR)))

## Synthèse à rédiger
- **Effet des proxys du temps** : ROC-AUC avec et sans `orders_last_7d` et `purchase_month` (cellule 5).
- **Meilleure approche** et sa stabilité mois par mois (cellules 5 et 6) ; classification directe contre régression + écart.
- **Performance à capacité égale** : précision, rappel et gain pour 5, 10 et 20 % de commandes signalées (cellule 7).
- **Rentabilité** : coût du retard manqué à partir duquel les alertes deviennent rentables (cellule 8).
- **Recommandation** : l'alerte est-elle à déployer, ou le score ne sert-il qu'à prioriser le suivi ?